# 01 · Locked SciQ splits and E0 data checks

**Runtime:** CPU is enough. **Time:** ~2 minutes.

What this does (protocol 3.2 and E0 data checks):
1. Downloads SciQ (`allenai/sciq`): official train 11,679 / validation 1,000 / test 1,000.
2. Shuffles the 4 options of each question with seed 0. The correct-answer position is a balanced
   random permutation, so each position holds the correct answer for 1/4 of the questions.
3. Leakage: exact duplicates of normalized question text across splits, and train questions with
   token-Jaccard > 0.9 to any dev/test question. Flagged train items are removed.
4. χ² test that the correct-answer position is uniform in each split.
5. Saves ID lists + SHA-256 hashes to `WORKSPACE/splits/`.

Running it again never overwrites the splits: it rebuilds them in a temp folder and checks that the
hashes match (reproducibility check).

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "YOUR_GITHUB_USERNAME/YOUR_REPO_NAME"              # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
!python scripts/build_splits.py --config configs/splits.yaml

### Review the flagged near-duplicate pairs by eye

In [ ]:
import json, os
chk = json.load(open(os.path.join(WORKSPACE, "splits", "sciq_data_checks.json")))
pairs = chk["near_duplicates_train_vs_devtest"]
print(f"{len(pairs)} flagged pairs; {sum(p['same_correct_answer'] for p in pairs)} share the correct answer\n")
for p in pairs[:30]:
    print(f"J={p['jaccard']:.2f} same_answer={p['same_correct_answer']}  {p['train_id']} vs {p['other_id']}")
    print("   train:", p["train_question"])
    print("   other:", p["other_question"])

### Look at a few formatted inputs (the exact text the LM sees)

In [ ]:
from optlm.data_sciq import load_items, format_option
for it in load_items("dev")[:2]:
    for j, o in enumerate(it["options"]):
        mark = "  <-- correct" if j == it["label"] else ""
        print(repr(format_option(it["question"], o)) + mark)
    print()

### Before moving on
Copy the three split hashes printed above into today's notebook entry (`notebook/TEMPLATE.md`).
Optionally download `WORKSPACE/splits/sciq_splits.json` into `data/splits/` in the repo and commit it.